In [2]:
# Configuración inicial de Selenium y Chrome en modo headless

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# Opciones de estabilidad para Docker
options = Options()
options.add_argument("--headless")          # Ejecutar sin abrir ventana
options.add_argument("--no-sandbox")        # Evita problemas de permisos
options.add_argument("--disable-dev-shm-usage")  # Manejo de memoria compartida
options.add_argument("--disable-gpu")       # Desactiva GPU (no necesaria en headless)

# Inicialización del driver con ChromeDriverManager
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)



In [ ]:
# Scraping de una sola página para validar extracción de datos

from selenium.webdriver.common.by import By

# Abrimos la página inicial
driver.get("https://books.toscrape.com/")

# Capturamos los productos de la primera página
productos = driver.find_elements(By.CLASS_NAME, "product_pod")
print("Cantidad de productos encontrados:", len(productos))

# Lista para almacenar resultados
datos = []

for p in productos:
    titulo = p.find_element(By.TAG_NAME, "h3").text
    precio = p.find_element(By.CLASS_NAME, "price_color").text
    datos.append({"titulo": titulo, "precio": precio})
    print(titulo, precio)  # 👈 feedback inmediato

# Mostrar resumen
print("Total de registros en la primera página:", len(datos))
print("Primeros 5 registros:", datos[:5])


In [ ]:
# Scraping con paginación: recorrer todas las páginas y capturar títulos y precios

import time

datos_totales = []

try:
    driver.get("https://books.toscrape.com/")
    while True:
        # Capturamos los productos de la página actual
        productos = driver.find_elements(By.CLASS_NAME, "product_pod")
        for p in productos:
            titulo = p.find_element(By.TAG_NAME, "h3").text
            precio = p.find_element(By.CLASS_NAME, "price_color").text
            datos_totales.append({"titulo": titulo, "precio": precio})
            print(titulo, precio)  # 👈 feedback inmediato

        # Intentamos pasar a la siguiente página
        try:
            boton_siguiente = driver.find_element(By.XPATH, "//a[contains(text(),'next')]")
            boton_siguiente.click()
            time.sleep(2)
        except:
            print("No hay más páginas")
            break

    # Resumen final
    print("Total de registros capturados:", len(datos_totales))
    print("Primeros 5 registros:", datos_totales[:5])

except Exception as e:
    print("Error durante el scraping:", e)


In [ ]:
# Conexión a MongoDB e inserción de los datos capturados

from pymongo import MongoClient

try:
    # Conexión al contenedor de MongoDB
    client = MongoClient("mongodb://bigdata_mongodb:27017/")
    db = client["test_db"]
    collection = db["libros"]

    # Insertar todos los registros
    collection.insert_many(datos_totales)

    # Confirmar cantidad de documentos insertados
    print("Datos insertados en MongoDB:", collection.count_documents({}))

except Exception as e:
    print("Error al insertar en MongoDB:", e)
